In [ ]:
# ------------------ Imports ------------------
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, roc_auc_score, classification_report
from xgboost import XGBRegressor, XGBClassifier
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ------------------ Load Data ------------------
df = pd.read_csv("../data/MachineLearningRating_v3.txt", sep="|")

# ------------------ Feature Engineering ------------------
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

features = ['Province', 'VehicleType', 'Gender', 'RegistrationYear',
            'CustomValueEstimate', 'kilowatts', 'Cylinders']
categorical = ['Province', 'VehicleType', 'Gender']
numerical = ['RegistrationYear', 'CustomValueEstimate', 'kilowatts', 'Cylinders']

# ------------------ Preprocessing Pipelines ------------------
numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical),
    ('cat', categorical_transformer, categorical)
])

# ------------------ Claim Severity Model ------------------
df_sev = df[df['TotalClaims'] > 0].copy()
X_sev = df_sev[features]
y_sev = df_sev['TotalClaims']
X_train_sev, X_test_sev, y_train_sev, y_test_sev = train_test_split(X_sev, y_sev, test_size=0.2, random_state=42)

models_sev = {
    # "Linear Regression": LinearRegression(),  ← remove or comment out
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42)
}


print("\n=== Claim Severity Models ===")
best_rmse = float("inf")
best_sev_model = None

for name, model in models_sev.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipeline.fit(X_train_sev, y_train_sev)
    preds = pipeline.predict(X_test_sev)
    mse = mean_squared_error(y_test_sev, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_sev, preds)
    print(f"\n{name}:\n  RMSE: {rmse:.2f}, R²: {r2:.3f}")

    
    if rmse < best_rmse:
        best_rmse = rmse
        best_sev_model = pipeline

# ------------------ Claim Probability Model ------------------
X_prob = df[features]
y_prob = df['HasClaim']
X_train_prob, X_test_prob, y_train_prob, y_test_prob = train_test_split(X_prob, y_prob, test_size=0.2, random_state=42)

cls_model = XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='logloss', random_state=42)
cls_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', cls_model)
])
cls_pipeline.fit(X_train_prob, y_train_prob)

cls_preds = cls_pipeline.predict(X_test_prob)
cls_probas = cls_pipeline.predict_proba(X_test_prob)[:, 1]

print("\n=== Claim Probability Model ===")
print(f"Accuracy: {accuracy_score(y_test_prob, cls_preds):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test_prob, cls_probas):.3f}")
print(classification_report(y_test_prob, cls_preds))

# ------------------ Risk-Based Premium Calculation ------------------
# Predict probabilities and severity
# Refit severity model on full X_prob (same as used during prediction)
full_sev_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(n_estimators=100, random_state=42))
])
full_sev_model.fit(X_sev, y_sev)

# Now prediction will match expected input shape
df['Severity_Pred'] = full_sev_model.predict(X_prob)

df['P_Claim'] = cls_pipeline.predict_proba(X_prob)[:, 1]
df['Severity_Pred'] = best_sev_model.predict(X_prob)

# Premium = P(Claim) × Severity + Loading + Margin
expense_loading = 200
profit_margin = 50
df['Risk_Based_Premium'] = (df['P_Claim'] * df['Severity_Pred']) + expense_loading + profit_margin

print("\n=== Sample Risk-Based Premiums ===")
print(df[['P_Claim', 'Severity_Pred', 'Risk_Based_Premium']].head())

# ------------------ SHAP Explainability (Regression Model) ------------------
print("\n=== SHAP: Feature Importance for Claim Severity ===")

# Get transformed data
X_all_transformed = best_sev_model.named_steps['preprocessor'].transform(X_prob)
explainer = shap.Explainer(best_sev_model.named_steps['model'], X_all_transformed)
shap_values = explainer(X_all_transformed)

shap.summary_plot(shap_values, features=preprocessor.get_feature_names_out(), max_display=10)

print("\n✅ Modeling & Prediction Complete.")




=== Claim Severity Models ===

Random Forest:
  RMSE: 41975.20, R²: -0.096

XGBoost:
  RMSE: 41438.43, R²: -0.068

=== Claim Probability Model ===
Accuracy: 0.997
ROC-AUC: 0.595
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    199439
           1       0.00      0.00      0.00       581

    accuracy                           1.00    200020
   macro avg       0.50      0.50      0.50    200020
weighted avg       0.99      1.00      1.00    200020



ValueError: Feature shape mismatch, expected: 20, got 21